# Aircraft XML to Excel Preprocessing

This notebook reads aircraft data from an XML file, extracts relevant fields, computes the tire contact area, and exports the results to an Excel file.

In [26]:
import xml.etree.ElementTree as ET
from openpyxl.styles import Font as XLFont, Alignment
from openpyxl.utils import get_column_letter
import pandas as pd
import numpy as np
from pathlib import Path

xml_path = Path("input_data/aircraft.xml")
tree = ET.parse(xml_path)
root = tree.getroot()

NS_F = "http://schemas.datacontract.org/2004/07/FaarFieldModel"
NS_A = "http://schemas.microsoft.com/2003/10/Serialization/Arrays"
NS_XSI = "http://www.w3.org/2001/XMLSchema-instance"

airplanes = root.find(f".//{{{NS_F}}}Airplanes")
assert airplanes is not None, "Could not find <Airplanes> in the XML."

def get_scalar(parent, tag, default=np.nan, cast=float):
    """Get text of a simple element <tag>value</tag> under parent."""
    el = parent.find(f"{{{NS_F}}}{tag}")
    if el is None:
        return default
    # handle xsi:nil="true"
    if el.attrib.get(f"{{{NS_XSI}}}nil", "").lower() == "true":
        return default
    if el.text is None:
        return default
    try:
        return cast(el.text.strip())
    except Exception:
        return default

def get_us(parent, tag, default=np.nan, cast=float):
    """Get the <us> child value under a unit-wrapped element <tag><si>..</si><us>..</us></tag>."""
    el = parent.find(f"{{{NS_F}}}{tag}")
    if el is None:
        return default
    us = el.find(f"{{{NS_F}}}us")
    if us is None or us.text is None:
        return default
    try:
        return cast(us.text.strip())
    except Exception:
        return default

In [27]:
rows = []
for ap in list(airplanes):
    # only keep the AirplaneInfo blocks
    if ap.tag != f"{{{NS_A}}}anyType":
        continue
    if ap.attrib.get(f"{{{NS_XSI}}}type") != "AirplaneInfo":
        continue

    name = ap.find(f"{{{NS_F}}}Name")
    name = name.text.strip() if (name is not None and name.text) else ""

    # Requested fields
    gw_lbs = get_us(ap, "_GrossWeight")          # "Gross Taxi Weight (lbs)"
    tire_area = get_us(ap, "TireArea", default=0.0)
    tire_len = get_us(ap, "TireLength", default=0.0)
    tire_wid = get_us(ap, "TireWidth", default=0.0)

    # TirePressureF is often nil; default to 0 per your requirement
    tire_pressure = get_us(ap, "Cp", default=0.0)
    mg_percent = get_scalar(ap, "MgPercent")
    mg_percent_pcn = get_scalar(ap, "MgPercentPCN")

    num_gear = get_scalar(ap, "NumberGear", cast=int)
    num_tracks = get_scalar(ap, "NumberTireTracks", cast=int, default=1.0)
    num_wheels = get_scalar(ap, "NumberWheels", cast=int)

    rows.append({
        "Airplane Name": name,
        "Gross Taxi Weight (lbs)": gw_lbs,
        "Tire Pressure (psi)": tire_pressure,
        "Percent GW on Gear": mg_percent,
        "MgPercentPCN": mg_percent_pcn,
        "Number Gear": num_gear,
        "Number Tire Tracks": num_tracks,
        "Number Wheels": num_wheels,
        "Tire Contact Width (in.)": tire_wid,
        "Tire Contact Length (in.)": tire_len,
        "Tire Contact Area (in.^2)": tire_area,
    })

df = pd.DataFrame(rows)

df["Number Tire Tracks"] = df["Number Tire Tracks"].replace(np.nan, 1) # Use 1 as default, to avoid division by 0 later

# Manual Entries
df = pd.concat([
    df,
    pd.DataFrame([{
        "Airplane Name": "ICT-B777-300ER",
        "Gross Taxi Weight (lbs)": 777000,
        "Tire Pressure (psi)": 200.0,
        "Percent GW on Gear": 0.525,
        "MgPercentPCN": 0.2660,
        "Number Gear": 3,
        "Number Tire Tracks": 4,
        "Number Wheels": 6,
        "Tire Contact Width (in.)": 13.39,
        "Tire Contact Length (in.)": 14.96,
        "Tire Contact Area (in.^2)": 200.0,
    }, {
        "Airplane Name": "ICT-A380-800",
        "Gross Taxi Weight (lbs)": 1239000,
        "Tire Pressure (psi)": 210.3,
        "Percent GW on Gear": 0.38,
        "MgPercentPCN": 0.19,
        "Number Gear": 3,
        "Number Tire Tracks": 4,
        "Number Wheels": 4,
        "Tire Contact Width (in.)": 14.17,
        "Tire Contact Length (in.)": 22.05,
        "Tire Contact Area (in.^2)": 270.00,
    }])
], ignore_index=True)


df["Load (lbs)"] = (df["Gross Taxi Weight (lbs)"] * df["MgPercentPCN"]) / df["Number Wheels"]
df["Load (N)"] = df["Load (lbs)"] * 4.44822
df["Tire Pressure (MPa)"] = df["Tire Pressure (psi)"] * 0.00689476
df["Tire Contact Length (mm)"] = df["Tire Contact Length (in.)"] * 25.4
df["Tire Contact Width (mm)"] = df["Tire Contact Width (in.)"] * 25.4
df["Tire Contact Area (cm.^2)"] = df["Tire Contact Area (in.^2)"] * 6.4516

# Categorization
df["Ribs (mm)"] = "50;35;40;90;40;35;50"
df["Load Factor"] = "0.24;0.08;0.08;0.20;0.08;0.08;0.24"
df["Stress Factor"] = "1.80;1.10;1.10;1.10;1.10;1.10;1.80"

# Optional: round the contact geometry to match your example output style
df["Tire Contact Width (in.)"] = df["Tire Contact Width (in.)"].round(1)
df["Tire Contact Length (in.)"] = df["Tire Contact Length (in.)"].round(1)
df["Tire Contact Area (in.^2)"] = df["Tire Contact Area (in.^2)"].round(1)


# Replace the value in Ribs(mm), Load Factor, and Stress Factor columns for this entry: ICT-A380-800
df.loc[df["Airplane Name"] == "ICT-A380-800", "Ribs (mm)"] = "70;45;90;45;70"
df.loc[df["Airplane Name"] == "ICT-A380-800", "Load Factor"] = "0.28;0.11;0.22;0.11;0.28"
df.loc[df["Airplane Name"] == "ICT-A380-800", "Stress Factor"] = "1.80;1.10;1.10;1.10;1.80"


num_formats = {
    "Gross Taxi Weight (lbs)": "0",
    "Tire Pressure (psi)": "0.0",
    "Percent GW on Gear": "0.0000",
    "MgPercentPCN": "0.0000",
    "Number Gear": "0",
    "Number Tire Tracks": "0",
    "Number Wheels": "0",
    "Tire Contact Width (in.)": "0.0",
    "Tire Contact Length (in.)": "0.0",
    "Tire Contact Area (in.^2)": "0.00",
    "Load (lbs)": "0",
    "Load (N)": "0",
    "Tire Pressure (MPa)": "0.000",
    "Tire Contact Length (mm)": "0",
    "Tire Contact Width (mm)": "0",
    "Tire Contact Area (cm.^2)": "0.00",
}

for c in num_formats:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")


out_path = Path("input_data/aircraft.xlsx")
with pd.ExcelWriter(out_path, engine="openpyxl") as writer:
    df.to_excel(writer, sheet_name="Aircraft", index=False)

    ws = writer.sheets["Aircraft"]

    base_font    = XLFont(name="Times New Roman", size=11)
    header_font  = XLFont(name="Times New Roman", size=11, bold=True)
    center_align = Alignment(horizontal="center", vertical="center")

    # Font + alignment
    for row in ws.iter_rows(min_row=1, max_row=ws.max_row,
                            min_col=1, max_col=ws.max_column):
        for cell in row:
            cell.font = header_font if cell.row == 1 else base_font
            cell.alignment = center_align

    # Map headers -> column index
    header_to_col = {
        ws.cell(row=1, column=col).value: col
        for col in range(1, ws.max_column + 1)
    }

    # Apply numeric formats (and coerce stray strings to numbers)
    for col_name, fmt in num_formats.items():
        col_idx = header_to_col.get(col_name)
        if not col_idx:
            continue
        for r in range(2, ws.max_row + 1):
            cell = ws.cell(row=r, column=col_idx)
            if isinstance(cell.value, str):
                try:
                    cell.value = float(cell.value)
                except Exception:
                    pass
            cell.number_format = fmt

    # Auto-fit widths
    for col_idx in range(1, ws.max_column + 1):
        max_len = 0
        for r in range(1, ws.max_row + 1):
            v = ws.cell(row=r, column=col_idx).value
            if v is None:
                continue
            if isinstance(v, float) and pd.isna(v):
                continue
            max_len = max(max_len, len(str(v)))
        ws.column_dimensions[get_column_letter(col_idx)].width = max_len + 2

In [28]:
# Visualize the first rows of the dataframe
df.head()

,Airplane Name,Gross Taxi Weight (lbs),Tire Pressure (psi),Percent GW on Gear,MgPercentPCN,Number Gear,Number Tire Tracks,Number Wheels,Tire Contact Width (in.),Tire Contact Length (in.),Tire Contact Area (in.^2),Load (lbs),Load (N),Tire Pressure (MPa),Tire Contact Length (mm),Tire Contact Width (mm),Tire Contact Area (cm.^2),Ribs (mm),Load Factor,Stress Factor
0,SWL-2,2000.0,30.0,1.0,1.0,1,1,1,7.3,11.7,66.7,2000.0,8896.44,0.206843,296.007901,185.004938,430.106650,50;35;40;90;40;35;50,0.24;0.08;0.08;0.20;0.08;0.08;0.24,1.80;1.10;1.10;1.10;1.10;1.10;1.80
1,SWL-5,5000.0,45.0,1.0,1.0,1,1,1,9.4,15.0,111.1,5000.0,22241.10,0.310264,382.144581,238.840363,716.844466,50;35;40;90;40;35;50,0.24;0.08;0.08;0.20;0.08;0.08;0.24,1.80;1.10;1.10;1.10;1.10;1.10;1.80
2,SWL-10,10000.0,50.0,1.0,1.0,1,1,1,12.6,20.2,200.0,10000.0,44482.20,0.344738,512.700760,320.437975,1290.320000,50;35;40;90;40;35;50,0.24;0.08;0.08;0.20;0.08;0.08;0.24,1.80;1.10;1.10;1.10;1.10;1.10;1.80
3,Single Wheel 2,2000.0,30.0,1.0,0.5,1,1,1,0.0,0.0,0.0,1000.0,4448.22,0.206843,0.000000,0.000000,0.000000,50;35;40;90;40;35;50,0.24;0.08;0.08;0.20;0.08;0.08;0.24,1.80;1.10;1.10;1.10;1.10;1.10;1.80
4,Single Wheel 5,5000.0,45.0,1.0,0.5,1,1,1,0.0,0.0,0.0,2500.0,11120.55,0.310264,0.000000,0.000000,0.000000,50;35;40;90;40;35;50,0.24;0.08;0.08;0.20;0.08;0.08;0.24,1.80;1.10;1.10;1.10;1.10;1.10;1.80
